# Legacy result reproduction

Runs the 60 heterophily and 2 Peptides-func experiments from `results/` using
`src/legacy_reproduction.py`. This notebook is an operator UI only: it pins the
last physical GPU, shows nested progress, and writes `reproduction_report.md`.

**Run All**, then leave the kernel running. Re-running resumes skipped artifacts
unless `RERUN` is set. Do not overwrite `results/` or `results_LRGB/`.

In [ ]:
import os
import subprocess
import sys

if "torch" in sys.modules:
    raise RuntimeError(
        "Torch was already imported. Restart the kernel and run this cell first."
    )

listing = subprocess.check_output(["nvidia-smi", "-L"], text=True)
gpus = [line for line in listing.splitlines() if line.startswith("GPU ")]
if not gpus:
    raise RuntimeError("nvidia-smi reported no GPUs")

LAST_GPU = str(len(gpus) - 1)
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = LAST_GPU
os.environ.setdefault("PYTHONHASHSEED", "25")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print(f"Using last GPU {LAST_GPU}: {gpus[-1]}")

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "legacy_reproduction.py").exists():
    ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from legacy_reproduction import (
    HETERO_CONFIG,
    HETERO_DATASETS,
    HETERO_MODELS,
    LRGB_CONFIG,
    LRGB_MODELS,
    generate_report,
    run_heterophily,
    run_lrgb,
    run_model,
    verify_h100,
)

SMOKE = False
RERUN = False
DATA_ROOT = ROOT / "data" / "legacy"
OUTPUT_ROOT = ROOT / "results_repro"
LRGB_OUTPUT_ROOT = ROOT / "results_LRGB_repro"
STATE_ROOT = ROOT / "reproduction_state"
REPORT_PATH = ROOT / "reproduction_report.md"
print(ROOT)

In [ ]:
import torch
import torch_geometric

gpu = verify_h100()
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"expected one visible GPU, got {torch.cuda.device_count()}")
print({
    "gpu": gpu["name"],
    "physical_index": gpu["physical_index"],
    "cuda_visible_devices": gpu["cuda_visible_devices"],
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "cuda": torch.version.cuda,
})

In [ ]:
from tqdm.auto import tqdm

if SMOKE:
    inner = tqdm(total=2, desc="smoke Minesweeper/MLP", leave=True)

    def on_epoch(epoch, epochs, **metrics):
        inner.set_postfix(val=f"{metrics.get('best_val', 0):.4f}")
        inner.update(1)

    path = run_heterophily(
        "Minesweeper",
        "MLP",
        OUTPUT_ROOT / "_smoke",
        DATA_ROOT,
        rerun=True,
        epochs=2,
        on_epoch=on_epoch,
    )
    inner.close()
    print(path)
else:
    print("Smoke disabled. Set SMOKE = True to run a 2-epoch check.")

In [ ]:
from tqdm.auto import tqdm

TOTAL_JOBS = len(HETERO_MODELS) * len(HETERO_DATASETS) + len(LRGB_MODELS)
outer = tqdm(total=TOTAL_JOBS, desc="experiments", unit="job")
inner = None


def close_inner():
    global inner
    if inner is not None:
        inner.close()
        inner = None


def start_inner(label, epochs, skipped):
    global inner
    close_inner()
    inner = tqdm(total=epochs, desc=label, leave=False)
    if skipped:
        inner.set_postfix(status="skip")
        inner.update(epochs)


def finish_job(_name, skipped=False):
    close_inner()
    outer.update(1)
    remaining = TOTAL_JOBS - outer.n
    outer.set_postfix(left=remaining, skipped=skipped)


def on_epoch(epoch, epochs, **metrics):
    if inner is None:
        return
    postfix = {}
    if "best_val" in metrics:
        postfix["val"] = f"{metrics['best_val']:.4f}"
    if "test_auroc" in metrics:
        postfix["test"] = f"{metrics['test_auroc']:.4f}"
    elif "test_ap" in metrics:
        postfix["test"] = f"{metrics['test_ap']:.4f}"
    inner.set_postfix(**postfix)
    inner.update(1)


def on_batch(batch_idx, n_batches, epoch, epochs):
    if inner is None:
        return
    inner.set_postfix(batch=f"{batch_idx + 1}/{n_batches}")


for model in HETERO_MODELS:
    def on_dataset(dataset_name, skipped=False, epochs=HETERO_CONFIG["epochs"], current=model):
        start_inner(f"{dataset_name}/{current}", epochs, skipped)

    run_model(
        model,
        OUTPUT_ROOT,
        DATA_ROOT,
        STATE_ROOT,
        rerun=RERUN,
        on_epoch=on_epoch,
        on_dataset=on_dataset,
        on_dataset_done=finish_job,
    )

run_lrgb(
    "GBDN+",
    LRGB_OUTPUT_ROOT,
    DATA_ROOT,
    rerun=RERUN,
    on_epoch=on_epoch,
    on_batch=on_batch,
    on_model=lambda name, skipped=False, epochs=LRGB_CONFIG["epochs"]: start_inner(
        f"Peptides-func/{name}", epochs, skipped
    ),
    on_model_done=finish_job,
)
close_inner()
outer.close()
print(f"Done. Heterophily JSON: {len(list(OUTPUT_ROOT.glob('*/*.json')))}/60")
print(f"Peptides-func JSON: {len(list(LRGB_OUTPUT_ROOT.glob('*.json')))}/2")

In [ ]:
import json
from IPython.display import Markdown, display

report_path = generate_report(
    ROOT / "results",
    OUTPUT_ROOT,
    ROOT / "results_LRGB",
    LRGB_OUTPUT_ROOT,
    REPORT_PATH,
)
display(Markdown(report_path.read_text(encoding="utf-8")))

rows = []
for path in sorted(OUTPUT_ROOT.glob("*/*.json")):
    reproduced = json.loads(path.read_text(encoding="utf-8"))
    original_path = ROOT / "results" / reproduced["dataset"] / f"{reproduced['model']}.json"
    original = json.loads(original_path.read_text(encoding="utf-8")) if original_path.exists() else {}
    rows.append({
        "dataset": reproduced["dataset"],
        "model": reproduced["model"],
        "acc_repro": reproduced.get("test_acc"),
        "acc_orig": original.get("test_acc"),
        "auroc_repro": reproduced.get("test_auroc"),
        "auroc_orig": original.get("test_auroc"),
    })
for path in sorted(LRGB_OUTPUT_ROOT.glob("*.json")):
    reproduced = json.loads(path.read_text(encoding="utf-8"))
    original_path = ROOT / "results_LRGB" / path.name
    original = json.loads(original_path.read_text(encoding="utf-8")) if original_path.exists() else {}
    rows.append({
        "dataset": reproduced["dataset"],
        "model": reproduced["model"],
        "ap_repro": reproduced.get("test_ap"),
        "ap_orig": original.get("test_ap"),
    })

try:
    import pandas as pd

    frame = pd.DataFrame(rows)
    if {"acc_repro", "acc_orig"} <= set(frame.columns):
        frame["acc_delta"] = frame["acc_repro"] - frame["acc_orig"]
    if {"auroc_repro", "auroc_orig"} <= set(frame.columns):
        frame["auroc_delta"] = frame["auroc_repro"] - frame["auroc_orig"]
    if {"ap_repro", "ap_orig"} <= set(frame.columns):
        frame["ap_delta"] = frame["ap_repro"] - frame["ap_orig"]
    display(frame)
except ImportError:
    print("pandas is not installed; metric tables are in the Markdown report above.")